# Intelligent Support Ticket Assistant

Reproducible demonstration of intent classification, category and priority assignment, historical-case retrieval, and evidence-based reply drafting for English and Arabic tickets.

Execute the cells in order. Python source comments, interface labels and technical documentation are in English. Customer input, training examples and customer-language reply data include English and Arabic. DemoMart is a fictional store used in the course dataset.

## 1. Project initialization

On a local Jupyter installation, run this notebook from the extracted repository. In Google Colab, upload the project ZIP when prompted. The project root contains `app.py` and `configs/project.json`.

In [ ]:
import os, sys, zipfile
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "configs/project.json").exists()), None)
if ROOT is None and IN_COLAB:
    existing_root = Path("/content/intelligent-support-ticket-assistant")
    if (existing_root / "configs/project.json").exists():
        ROOT = existing_root
if ROOT is None and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    archive = next((name for name in uploaded if name.lower().endswith(".zip")), None)
    if archive is None:
        raise ValueError("Upload the project ZIP archive.")
    destination = Path("/content").resolve()
    with zipfile.ZipFile(archive) as z:
        if any(not (destination / item.filename).resolve().is_relative_to(destination) for item in z.infolist()):
            raise ValueError("Unsafe archive path.")
        z.extractall(destination)
    ROOT = next((p.parent.parent for p in destination.glob("*/configs/project.json")), None)
if ROOT is None:
    raise FileNotFoundError("Extract the project and run the notebook from its directory.")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project directory:", ROOT.name)

## 2. Dependencies

The setup cell checks installed and already-loaded scientific-library versions. If pip changes them in Colab, execution pauses and requests **Runtime → Restart session**. After restart, execute from the first cell again; the extracted project is reused. Imports occur only after setup confirms that a restart is not required.


In [ ]:
import subprocess, importlib.util
from importlib import metadata
DEPENDENCIES_READY = False
critical_packages = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn"}

def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

before = {name: installed_version(package) for name, package in critical_packages.items()}
loaded = {name: getattr(sys.modules.get(name), "__version__", None) for name in critical_packages}
if os.environ.get("PROJECT_SKIP_INSTALL") != "1":
    if importlib.util.find_spec("torch") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.14.1", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
after = {name: installed_version(package) for name, package in critical_packages.items()}
changed = [name for name in critical_packages if before[name] != after[name]]
stale = [name for name in critical_packages if loaded[name] is not None and loaded[name] != after[name]]
if IN_COLAB and (changed or stale):
    print("Dependencies installed. Restart the Colab session before importing the scientific libraries.")
    print("Select Runtime > Restart session, then run from the first cell again.")
    raise RuntimeError("Setup completed: a session restart is required. This is an intentional setup pause.")
DEPENDENCIES_READY = True
print("Dependency setup complete. Continue to the import cell.")


In [ ]:
if not globals().get("DEPENDENCIES_READY", False):
    raise RuntimeError("Complete dependency setup and restart the session if requested.")
import torch, pandas as pd, numpy as np, json
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
print("NumPy:", np.__version__, "pandas:", pd.__version__)


## 3. Rebuild option

The repository includes prepared data and trained LR classifiers. Set `REBUILD=True` to regenerate data splits, train the classifiers and evaluate them. Frozen e5 weights remain unchanged during LR training.

In [ ]:
REBUILD = False
if REBUILD:
    for module in ["src.prepare_data", "src.train", "src.evaluate"]:
        subprocess.run([sys.executable, "-m", module], check=True)
else:
    assert (ROOT / "artifacts/e5_bilingual.joblib").exists(), "Run python -m src.train first."
print("Data and classifiers are available.")

## 4. Data overview

Training fits the head, validation selects C and confidence thresholds, and held-out tests compare systems. Historical resolved tickets form a disjoint retrieval index.

In [ ]:
from src.common import DATA, CONFIG
manifest = json.loads((DATA / "manifest.json").read_text())
display(pd.DataFrame(manifest["sizes"].items(), columns=["split", "rows"]))
train_en = pd.read_csv(DATA / "train.csv")
train_ar = pd.read_csv(DATA / "train_ar.csv")
display(pd.concat([train_en[["text", "intent"]].head(2), train_ar[["text", "intent"]].head(2)], ignore_index=True))
print("Intent classes:", train_en.intent.nunique())

## 5. Pipeline initialization

The default path drafts from retrieved source text or a source-linked Arabic summary. Setting `USE_LLM=True` enables the local Qwen evidence-sentence selector. Both paths return the same output contract.

In [ ]:
from src.pipeline import TicketAssistant
USE_LLM = False
assistant = TicketAssistant(use_llm=USE_LLM)
print("Representation device:", assistant.embedder.device)

## 6. Token and vector representation

The tokenizer produces input IDs and masks. Normalized e5 representations have 384 dimensions. The dot product of normalized vectors is cosine similarity.

In [ ]:
texts = ["How do I delete my account?", "كيف أحذف حسابي؟", "Where is my order?"]
tokenizer = assistant.embedder.model.tokenizer
for text in texts[:2]:
    encoded = tokenizer("query: " + text, truncation=True, max_length=256)
    print("Input:", text)
    print("Token IDs:", encoded["input_ids"])
    print("Attention mask:", encoded["attention_mask"])
vectors = assistant.embedder.encode(texts)
print("Representation shape:", vectors.shape)
display(pd.DataFrame(vectors @ vectors.T, index=texts, columns=texts).round(3))

## 7. End-to-end sample

The result contains an accepted intent or a review route, historical cases, knowledge-base text, citations and a draft. `predicted_intent` preserves the highest-scoring class even when the confidence gate rejects it.

In [ ]:
ticket_text = "How do I delete my account?"  # Replace with a support-ticket input.
result = assistant(ticket_text)
display(pd.DataFrame([{key: result[key] for key in ["intent", "predicted_intent", "confidence", "category", "priority", "route"]}]))
print("Draft reply:", result["reply"])
print("Source IDs:", result["citations"])
display(pd.DataFrame(result["similar_tickets"])[["ticket_id", "intent", "text", "score"]])
print(json.dumps(result, ensure_ascii=False, indent=2))

## 8. Bilingual and out-of-domain samples

These examples exercise an Arabic accepted ticket, an uncertain Arabic ticket and an unrelated English question. Results are measured outputs; coverage and scope rejection are limited.

In [ ]:
rows = []
for text in ["أرغب في حذف حسابي نهائياً", "تم خصم المبلغ مرتين من بطاقتي!!", "What is your company's stock price?"]:
    output = assistant(text)
    rows.append({"input": text, **{key: output[key] for key in ["intent", "predicted_intent", "confidence", "route", "reply_status"]}})
    print("Input:", text)
    print("Draft:", output["reply"], "\n")
display(pd.DataFrame(rows))

## 9. Held-out evaluation

All classifiers use the same 513-message held-out test. Raw macro-F1, confidence acceptance coverage and accepted accuracy measure different aspects of performance. The retrieval table distinguishes development and authored test partitions.

In [ ]:
metrics = pd.read_csv(ROOT / "eval/classification_metrics.csv")
display(metrics[metrics["slice"].isin(["all", "en", "ar"])][["model", "slice", "n", "macro_f1", "coverage", "accepted_accuracy", "classifier_p95_ms"]].round(4))
display(pd.read_csv(ROOT / "eval/retrieval_metrics.csv").round(4))
display(pd.read_csv(ROOT / "eval/error_analysis.csv")[["text", "y_true", "y_pred", "accepted", "cause"]])

## 10. Interactive interface

The local interface runs on localhost. In Colab, `share=True` creates a temporary public URL while the notebook session is active. The interface displays drafts for support-agent review.

In [ ]:
from app import build_app
demo = build_app(assistant)
if os.environ.get("PROJECT_NO_LAUNCH") != "1":
    demo.launch(share=IN_COLAB, prevent_thread_lock=True)
else:
    print("Interface constructed; HTTP behavior is verified separately.")